# Source audio analysis
This notebook reads the preserved Hugging Face source snapshot under `sources/`.
For model training, use the organized `train.csv` and `test.csv` at the dataset root; see `dataset_splits.ipynb`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import IPython.display as ipd
from pathlib import Path
import wave

def sample_rate(path):
    with wave.open(str(path), "rb") as audio: 
        return audio.getframerate()

data_dir = next(
    (path for path in (Path("data"), Path("../data")) if (path / "karakalpak-audio-dataset-hf/sources/train.csv").is_file()),
    None,
)
hf_dir = data_dir / "karakalpak-audio-dataset-hf" / "sources"
data_files = sorted(
    (hf_dir / "data").glob("*.wav"),
    key=lambda path: (sample_rate(path), path.name),
)
audio_dir = hf_dir / "data_16k"
audio_16k_by_name = {path.name: path for path in audio_dir.glob("*.wav")}
if {path.name for path in data_files} != set(audio_16k_by_name):
    raise ValueError("data and data_16k must contain the same filenames")
audio_files = [audio_16k_by_name[path.name] for path in data_files]
len(audio_files)

In [ ]:
import pandas as pd
import wave
from datasets import Audio, Dataset

def audio_duration(path):
    with wave.open(str(path), "rb") as audio:
        return audio.getnframes() / audio.getframerate()

frames = [pd.read_csv(hf_dir / f"{split}.csv") for split in ("train", "test")]

frame = pd.concat(frames, ignore_index=True).set_index("filename", verify_integrity=True)

audio_names = [path.name for path in data_files]
if set(frame.index) != set(audio_names):
    raise ValueError("CSV filenames must match the audio filenames")

    
frame = frame.loc[audio_names].reset_index()
frame.insert(0, "audio_index", range(len(frame)))
frame["duration"] = [audio_duration(path) for path in audio_files]



## Estimated SNR for every recording

The estimate compares energy in speech regions with energy in quiet regions. A recording without both regions gets a missing value.


In [ ]:
def estimate_snr(path):
    audio, sr = librosa.load(path, sr=None)
    frame_length = round(0.025 * sr)
    hop_length = round(0.010 * sr)
    intervals = librosa.effects.split(audio, top_db=30, frame_length=frame_length, hop_length=hop_length)
    active = np.zeros(len(audio), dtype=bool)
    for start, end in intervals:
        active[start:end] = True
    if not active.any() or active.all():
        return np.nan
    speech_power = np.mean(audio[active] ** 2)
    quiet_power = np.mean(audio[~active] ** 2)
    return 10 * np.log10((speech_power + 1e-12) / (quiet_power + 1e-12))


In [ ]:
frame["snr_original"] = [estimate_snr(path) for path in data_files]
frame[["audio_index", "filename", "snr_original"]].head()


In [ ]:
frame["snr_16k"] = [estimate_snr(path) for path in audio_files]
frame[["audio_index", "filename", "snr_original", "snr_16k"]].head()


In [ ]:
manifest_path = hf_dir / "data.csv"
partial_manifest = manifest_path.with_suffix(".csv.partial")
frame.to_csv(partial_manifest, index=False)
partial_manifest.replace(manifest_path)


In [ ]:
frame["audio_original"] = [{"path": str(path.resolve()), "bytes": None} for path in data_files]
frame["audio"] = [{"path": str(path.resolve()), "bytes": None} for path in audio_files]
dataset = Dataset.from_pandas(frame, preserve_index=False).cast_column("audio_original", Audio(decode=False)).cast_column("audio", Audio(decode=False))
dataset.select_columns(["audio_index", "filename", "duration", "snr_original", "snr_16k"])


In [ ]:
selected_index = 9
selected = dataset[selected_index]
selected["audio_index"], selected["filename"], selected["snr_original"], selected["snr_16k"], selected["sentence_latin"]


## Before: original recording

The two sections use the same file. Fbank is log Mel energy. The VAD uses an energy threshold, so the SNR from active and quiet regions is only an estimate.

In [ ]:
before, before_sr = librosa.load(selected["audio_original"]["path"], sr=None)
print(f"{selected['filename']} — {before_sr} Hz")
ipd.Audio(before, rate=before_sr)

In [ ]:
before_fft = round(0.025 * before_sr)
before_hop = round(0.010 * before_sr)
before_stft = np.abs(librosa.stft(before, n_fft=before_fft, hop_length=before_hop))
plt.figure(figsize=(10, 3))
librosa.display.specshow(librosa.amplitude_to_db(before_stft, ref=np.max), sr=before_sr, hop_length=before_hop, x_axis="time", y_axis="log")
plt.title("STFT: original")
plt.colorbar(format="%+2.0f dB")
plt.tight_layout()
plt.show()

In [ ]:
before_mel = librosa.feature.melspectrogram(y=before, sr=before_sr, n_fft=before_fft, hop_length=before_hop, n_mels=80)
before_mel_db = librosa.power_to_db(before_mel, ref=np.max)

plt.figure(figsize=(10, 3))
librosa.display.specshow(before_mel_db, sr=before_sr, hop_length=before_hop, x_axis="time", y_axis="mel", vmin=-80, vmax=0)
plt.ylim(0, 8000)
plt.title("Mel spectrogram: original")
plt.colorbar(format="%+2.0f dB")
plt.tight_layout()
plt.show()


In [ ]:
before_fbank = np.log(before_mel + 1e-10)
plt.figure(figsize=(10, 3))

librosa.display.specshow(before_fbank, sr=before_sr, hop_length=before_hop, x_axis="time", y_axis="mel")

plt.title("Fbank: original")
plt.colorbar(label="log power")
plt.tight_layout()
plt.show()

In [ ]:
before_intervals = librosa.effects.split(before, top_db=30, frame_length=before_fft, hop_length=before_hop)
before_active = np.zeros(len(before), dtype=bool)


for start, end in before_intervals:
    before_active[start:end] = True
plt.figure(figsize=(10, 2))
librosa.display.waveshow(before, sr=before_sr)


for start, end in before_intervals:
    plt.axvspan(start / before_sr, end / before_sr, color="tab:green", alpha=0.25)
 
     
plt.title("Energy VAD: original")
plt.tight_layout()
plt.show()

In [ ]:
if before_active.any() and (~before_active).any():
    before_snr = 10 * np.log10((np.mean(before[before_active] ** 2) + 1e-12) / (np.mean(before[~before_active] ** 2) + 1e-12))
else:
    before_snr = np.nan
    
print(f"Estimated SNR: {before_snr:.1f} dB")

## After: 16 kHz recording

In [ ]:
after, after_sr = librosa.load(selected["audio"]["path"], sr=None)
print(f"{selected['filename']} — {after_sr} Hz")
ipd.Audio(after, rate=after_sr)

In [ ]:
after_fft = round(0.025 * after_sr)
after_hop = round(0.010 * after_sr)
after_stft = np.abs(librosa.stft(after, n_fft=after_fft, hop_length=after_hop))

plt.figure(figsize=(10, 3))
librosa.display.specshow(librosa.amplitude_to_db(after_stft, ref=np.max), sr=after_sr, hop_length=after_hop, x_axis="time", y_axis="log")
plt.title("STFT: 16 kHz")
plt.colorbar(format="%+2.0f dB")
plt.tight_layout()
plt.show()


In [ ]:
after_mel = librosa.feature.melspectrogram(y=after, sr=after_sr, n_fft=after_fft, hop_length=after_hop, n_mels=80)
after_mel_db = librosa.power_to_db(after_mel, ref=np.max)

plt.figure(figsize=(10, 3))
librosa.display.specshow(after_mel_db, sr=after_sr, hop_length=after_hop, x_axis="time", y_axis="mel", vmin=-80, vmax=0)
plt.ylim(0, 8000)
plt.title("Mel spectrogram: 16 kHz")
plt.colorbar(format="%+2.0f dB")
plt.tight_layout()
plt.show()


In [ ]:
after_fbank = np.log(after_mel + 1e-10)
plt.figure(figsize=(10, 3))

librosa.display.specshow(after_fbank, sr=after_sr, hop_length=after_hop, x_axis="time", y_axis="mel")

plt.title("Fbank: 16 kHz")
plt.colorbar(label="log power")
plt.tight_layout()
plt.show()

In [ ]:
after_intervals = librosa.effects.split(after, top_db=30, frame_length=after_fft, hop_length=after_hop)
after_active = np.zeros(len(after), dtype=bool)

for start, end in after_intervals:
    after_active[start:end] = True
plt.figure(figsize=(10, 2))
librosa.display.waveshow(after, sr=after_sr)

for start, end in after_intervals:
    plt.axvspan(start / after_sr, end / after_sr, color="tab:green", alpha=0.25)
    
plt.title("Energy VAD: 16 kHz")
plt.tight_layout()
plt.show()

In [ ]:
if after_active.any() and (~after_active).any():
    after_snr = 10 * np.log10((np.mean(after[after_active] ** 2) + 1e-12) / (np.mean(after[~after_active] ** 2) + 1e-12))
else:
    after_snr = np.nan
print(f"Estimated SNR: {after_snr:.1f} dB")
print(f"Change: {after_snr - before_snr:+.1f} dB")